# SNOM Connection & Movement Test
Tests `nea_tools` connection, position read-back, and XY tip movement via the `context.Logic.MoveTipPosition` API.

## 0 — Configuration

In [ ]:
HOST        = 'nea-server'   # change to IP if DNS does not resolve
PATH_TO_DLL = ''             # leave empty unless nea_tools requires a specific path

SPEED_UM_S  = 0.2            # tip movement speed in µm/s

# Test positions in µm — origin is (50, 50) µm
# Visits a small 1 µm square around the default scan origin and returns home.
ORIGIN_UM = (50.0, 50.0)
TEST_POSITIONS_UM = [
    ORIGIN_UM,           # home
    (51.0, 50.0),        # +1 µm X
    (51.0, 51.0),        # +1 µm X, +1 µm Y
    (50.0, 51.0),        # +1 µm Y
    ORIGIN_UM,           # back to origin
]

## 1 — Imports

In [ ]:
import asyncio
from time import sleep

import nest_asyncio
import numpy as np
import nea_tools

print('nea_tools version:', getattr(nea_tools, '__version__', 'unknown'))

## 2 — Connect to nea-server

In [ ]:
loop = asyncio.get_event_loop()
nest_asyncio.apply(loop)

print(f'Connecting to {HOST} ...')
loop.run_until_complete(nea_tools.connect(HOST, fingerprint=None, path_to_dll=PATH_TO_DLL))
print('Connected.')

## 3 — Import neaspec and Nea.Client.SharedDefinitions (injected after connect)

In [ ]:
from neaspec import context
import Nea.Client.SharedDefinitions as nea

print('context :', context)
print('nea     :', nea)

## 4 — Read current position\n\n> **Note:** position readback API not yet confirmed with developer. Skip this cell or ask Vincent for the correct call.

In [ ]:
# TODO: ask Vincent for the correct position readback call
# pos = context.???
# print(f'X={pos.X}  Y={pos.Y}  Z={pos.Z}')

## 5 — Movement helper
Mirrors the developer snippet exactly: attaches event callbacks, fires `MoveTipPosition`, polls until the move completes.

In [ ]:
do_wait = False

def on_tip_position_moved(sender, args):
    print(f'Arrived at {args.Point}')
    global do_wait
    do_wait = False

def on_tip_position_moving(sender, args):
    print(f'Moving to {args.Point}')
    global do_wait
    do_wait = True

context.Logic.TipPositionMoved  += on_tip_position_moved
context.Logic.TipPositionMoving += on_tip_position_moving

def move_tip(x_um: float, y_um: float, speed_um_s: float = SPEED_UM_S) -> None:
    """Move tip to (x_um, y_um) and block until done."""
    move_args = nea.MoveTipPositionArgs(
        nea.Geometry.Point2D(x_um, y_um),
        speed_um_s / 1000.0,
    )
    context.Logic.MoveTipPosition.Execute(move_args)
    sleep(0.1)
    while do_wait:
        sleep(0.1)
    sleep(0.2)

print('move_tip() defined.')

## 6 — Movement test

In [ ]:
import time

results = []

for (x_um, y_um) in TEST_POSITIONS_UM:
    print(f'\nTarget: ({x_um}, {y_um}) µm')
    t0 = time.time()
    move_tip(x_um, y_um)
    dt = time.time() - t0
    print(f'  Move completed in {dt:.2f} s')
    results.append(dict(x_um=x_um, y_um=y_um, dt_s=dt))

print('\nMovement test complete.')

## 7 — Summary table

In [ ]:
import pandas as pd
df = pd.DataFrame(results)
df

## 8 — Read optical & mechanical amplitudes

In [ ]:
import neaspec.stream as stream

o_amp = np.empty(6)
m_amp = np.empty(6)
o_phase = np.empty(6)
m_phase = np.empty(6)

with stream.Stream() as s:
    for harmonic in range(6):
        o_amp[harmonic] = context.Microscope.Py.OpticalAmplitude(harmonic)
        m_amp[harmonic] = context.Microscope.Py.MechanicalAmplitude(harmonic)
        try:
            o_phase[harmonic] = s.data[f"O{harmonic}P"][-1]
        except Exception as e:
            o_phase[harmonic] = float('nan')
            print(f"Warning: could not read phase for O{harmonic}P: {e}")
        try:
            m_phase[harmonic] = s.data[f"M{harmonic}P"][-1]
        except Exception as e:
            m_phase[harmonic] = float('nan')
            print(f"Warning: could not read phase for M{harmonic}P: {e}")

print(f"{'Harmonic':>10}  {'O_Amp':>12}  {'O_Phase':>12}  {'M_Amp':>12}  {'M_Phase':>12}")
print('-' * 65)
for h in range(6):
    print(f"{h:>10}  {o_amp[h]:>12.6f}  {o_phase[h]:>12.6f}  {m_amp[h]:>12.6f}  {m_phase[h]:>12.6f}")

## 9 — Disconnect

In [ ]:
try:
    nea_tools.disconnect()
    print('Disconnected.')
except Exception as e:
    print(f'Disconnect error (ignored): {e}')